In [22]:
import torch
import torch.nn as nn
import torch.nn.functional as F 
import random
import numpy as np
from torch_geometric.transforms import NormalizeFeatures

seed = 42

torch.manual_seed(seed)
random.seed(seed)
np.random.seed(seed)

torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [3]:
from torch_geometric.datasets import Planetoid

In [23]:

dataset = Planetoid(root="data", name="Cora", transform=NormalizeFeatures())
data = dataset[0]

In [5]:
class GraphSAGESampler:
  def __init__(self, adj):
    self.adj = adj
 
  def sample_neighbors(self, nodes, num_samples):

    neigh = self.adj[nodes]  # [B, max_degree]

    if neigh.shape[1] >= num_samples:
        perm = torch.rand(neigh.shape).argsort(dim=1)
        neigh = torch.gather(neigh, 1, perm[:, :num_samples])
    else:
        idx = torch.randint(0, neigh.shape[1], (nodes.shape[0], num_samples))
        neigh = torch.gather(neigh, 1, idx)

    return neigh
  
  def sample(self, batch_nodes, num_samples_list):
    layers = [batch_nodes]
    current = batch_nodes


    for num_samples in num_samples_list:
      neigh = self.sample_neighbors(current, num_samples)

      current = neigh.reshape(-1) # flattening

      layers.append(current)

    return layers

In [17]:
class MeanAggregator(nn.Module):

  def __init__(self, input_dim, output_dim, dropout=0.0, bias=False, concat=False):
    super().__init__()

    self.dropout = dropout
    self.concat = concat

    self.self_weights = nn.Linear(input_dim, output_dim, bias=bias)
    self.neigh_weights = nn.Linear(input_dim, output_dim, bias=bias)

    self.act = nn.ReLU()

  def forward(self, self_vecs, neigh_vecs):
    """
    self_vecs:  [B, D]
    neigh_vecs: [B, S, D]
    """

    self_vecs = F.dropout(self_vecs, self.dropout, self.training)
    neigh_vecs = F.dropout(neigh_vecs, self.dropout, self.training)

    neigh_mean = neigh_vecs.mean(dim=1)   # [B, D]

    self_part = self.self_weights(self_vecs)
    neigh_part = self.neigh_weights(neigh_mean)

    if self.concat:
        out = torch.cat([self_part, neigh_part], dim=1)
    else:
        out = self_part + neigh_part

    return self.act(out)


class MaxPoolAggregator(nn.Module):
  def __init__(self, input_dim, output_dim, hidden_dim = 256):
    super().__init__()
    self.project_to_hidden = nn.Linear(input_dim, hidden_dim)
    self.project_back = nn.Linear(hidden_dim, output_dim)
    self.project_self_vecs = nn.Linear(input_dim, output_dim)
    self.act = nn.ReLU()

  def forward(self, self_vecs, neigh_vecs):
    '''
    project -> pool -> project_back -> concat
    '''
    neigh_projected = self.act(self.project_to_hidden(neigh_vecs))
    self_projected = self.project_self_vecs(self_vecs)
    neighs_pooled = torch.max(neigh_projected, dim = 1)
    neighs_pooled_projected_back = self.project_back(neighs_pooled)

    concatinated = self_projected + neighs_pooled_projected_back

    return self.act(concatinated)
    

In [7]:
class GraphSAGE(nn.Module):

    def __init__(self, features, sampler, aggregators, num_samples):

        super().__init__()

        self.features = features
        self.sampler = sampler
        self.aggregators = nn.ModuleList(aggregators)
        self.num_samples = num_samples

    def forward(self, nodes):

        layers = self.sampler.sample(nodes, self.num_samples)

        hidden = [self.features[layer] for layer in layers]

        num_layers = len(self.aggregators)

        for layer in range(num_layers):

            aggregator = self.aggregators[layer]
            next_hidden = []

            for hop in range(num_layers - layer):

                parent = hidden[hop]
                neigh = hidden[hop + 1]

                num_neigh = self.num_samples[hop]   # ← FIX

                feat_dim = neigh.shape[1]

                neigh = neigh.view(parent.shape[0], num_neigh, feat_dim)

                out = aggregator(parent, neigh)

                next_hidden.append(out)

            hidden = next_hidden

        return hidden[0]

In [34]:
class SAGEClassifier(nn.Module):

    def __init__(self, graph_sage, hidden_dim, num_classes,dropout = 0.5):

        super().__init__()

        self.graph_sage = graph_sage
        self.classifier = nn.Linear(hidden_dim, num_classes)
        self.dropout = nn.Dropout(dropout)


    def forward(self, nodes):
        h = self.graph_sage(nodes)
        h = F.normalize(h, p=2, dim = 1)
        h = self.dropout(h)
        return self.classifier(h)

In [ ]:
import random

def build_adj_matrix(edge_index, num_nodes, max_degree):

    adj = torch.zeros((num_nodes, max_degree), dtype=torch.long)

    neighbors = [[] for _ in range(num_nodes)]

    src, dst = edge_index

    for u, v in zip(src.tolist(), dst.tolist()):
        neighbors[u].append(v)
        neighbors[v].append(u)

    for i in range(num_nodes):

        neigh = neighbors[i]

        if len(neigh) == 0:
            continue

        if len(neigh) >= max_degree:
            neigh = random.sample(neigh, max_degree)
        else:
            while len(neigh) < max_degree:
                neigh.append(random.choice(neigh))

        adj[i] = torch.tensor(neigh)

    return adj

adj = build_adj_matrix(data.edge_index, data.num_nodes, max_degree=336)
adj.shape

torch.Size([2708, 300])

In [29]:
adj_list = [[] for _ in range(data.num_nodes)]

src, dst = data.edge_index
for u,v in zip(src.tolist(), dst.tolist()):
  adj_list[u].append(v)
  adj_list[v].append(u)
adj_list

[[633, 1862, 2582, 633, 1862, 2582],
 [2, 652, 654, 2, 652, 654],
 [1, 1, 332, 1454, 1666, 1986, 332, 1454, 1666, 1986],
 [2544, 2544],
 [1016, 1256, 1761, 2175, 2176, 1016, 1256, 1761, 2175, 2176],
 [1629, 1659, 2546, 1629, 1659, 2546],
 [373, 1042, 1416, 1602, 373, 1042, 1416, 1602],
 [208, 208],
 [269, 281, 1996, 269, 281, 1996],
 [723, 2614, 723, 2614],
 [476, 2545, 476, 2545],
 [1655, 1839, 1655, 1839],
 [1001, 1318, 2661, 2662, 1001, 1318, 2661, 2662],
 [1701, 1810, 1701, 1810],
 [158, 2034, 2075, 2077, 2668, 158, 2034, 2075, 2077, 2668],
 [1090, 1093, 1271, 2367, 1090, 1093, 1271, 2367],
 [970, 1632, 2444, 2642, 970, 1632, 2444, 2642],
 [24, 927, 1315, 1316, 2140, 24, 927, 1315, 1316, 2140],
 [139, 1560, 1786, 2082, 2145, 139, 1560, 1786, 2082, 2145],
 [1939, 1939],
 [1072, 2269, 2270, 2374, 2375, 1072, 2269, 2270, 2374, 2375],
 [1043, 2310, 1043, 2310],
 [39, 1234, 1702, 1703, 2238, 39, 1234, 1702, 1703, 2238],
 [2159, 2159],
 [17, 17, 201, 598, 1636, 1701, 2139, 2141, 201, 598

In [30]:
sampler = GraphSAGESampler(adj)

agg1 = MeanAggregator(1433, 128)
agg2 = MeanAggregator(128, 128)

sage = GraphSAGE(
    features=data.x,
    sampler=sampler,
    aggregators=[agg1, agg2],
    num_samples=[25, 10]
)

classifier_cls = SAGEClassifier(
    sage,
    hidden_dim=128,
    num_classes=dataset.num_classes
)
nn.utils.clip_grad_norm_(classifier_cls.parameters(), 5)

tensor(0.)

In [39]:
optimizer = torch.optim.Adam(classifier_cls.parameters(), lr=0.01, weight_decay= 5e-4)
criterion = nn.CrossEntropyLoss()

def train():

    classifier_cls.train()

    optimizer.zero_grad()

    train_nodes = torch.where(data.train_mask)[0]

    logits = classifier_cls(train_nodes)

    loss = criterion(logits, data.y[train_nodes])

    loss.backward()
    
    optimizer.step()

    return loss.item()

In [40]:
for epoch in range(200):
  loss = train()
  if epoch % 10 == 0:
    print(f"Epoch {epoch}, loss: {loss: }")

Epoch 0, loss:  0.14317895472049713
Epoch 10, loss:  0.00359712284989655
Epoch 20, loss:  0.015997014939785004
Epoch 30, loss:  0.016346748918294907
Epoch 40, loss:  0.010979006066918373
Epoch 50, loss:  0.0109394621104002
Epoch 60, loss:  0.010169906541705132
Epoch 70, loss:  0.012674776837229729
Epoch 80, loss:  0.013319249264895916
Epoch 90, loss:  0.014374833554029465
Epoch 100, loss:  0.01107258815318346
Epoch 110, loss:  0.013609240762889385
Epoch 120, loss:  0.013940776698291302
Epoch 130, loss:  0.011327926069498062
Epoch 140, loss:  0.0177998635917902
Epoch 150, loss:  0.013048968277871609
Epoch 160, loss:  0.01218654215335846
Epoch 170, loss:  0.011390033178031445
Epoch 180, loss:  0.015561981126666069
Epoch 190, loss:  0.01177505124360323


In [41]:
from sklearn.metrics import classification_report

def test(mask):

    classifier_cls.eval()

    with torch.no_grad():

        nodes = torch.where(mask)[0]

        logits = classifier_cls(nodes)

        pred = logits.argmax(dim=1)

        correct = (pred == data.y[nodes]).sum().item()

        acc = correct / nodes.shape[0]

    return acc

def classification_metrics(mask):

    classifier_cls.eval()

    with torch.no_grad():

        nodes = torch.where(mask)[0]

        logits = classifier_cls(nodes)

        preds = logits.argmax(dim=1).cpu()
        labels = data.y[nodes].cpu()

    print(classification_report(labels, preds))

train_acc = test(data.train_mask)
val_acc = test(data.val_mask)
test_acc = test(data.test_mask)

print("Train Accuracy:", train_acc)
print("Validation Accuracy:", val_acc)
print("Test Accuracy:", test_acc)
classification_metrics(data.test_mask)

Train Accuracy: 1.0
Validation Accuracy: 0.73
Test Accuracy: 0.752
              precision    recall  f1-score   support

           0       0.63      0.68      0.65       130
           1       0.77      0.89      0.83        91
           2       0.88      0.81      0.84       144
           3       0.86      0.74      0.79       319
           4       0.73      0.75      0.74       149
           5       0.72      0.74      0.73       103
           6       0.58      0.81      0.68        64

    accuracy                           0.76      1000
   macro avg       0.74      0.77      0.75      1000
weighted avg       0.77      0.76      0.76      1000

